In [1]:
!git clone https://github.com/andyzoujm/representation-engineering.git

Cloning into 'representation-engineering'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 299 (delta 105), reused 83 (delta 83), pack-reused 123 (from 1)
Receiving objects: 100% (299/299), 614.06 KiB | 15.75 MiB/s, done.
Resolving deltas: 100% (129/129), done.


In [2]:
%cd representation-engineering/

!pip install -e .
!pip install -U bitsandbytes>=0.46.1

/content/representation-engineering
Obtaining file:///content/representation-engineering
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for repe (pyproject.toml) ... done
  Created wheel for repe: filename=repe-0.1.4-0.editable-py3-none-any.whl size=5588 sha256=04bb015b7d1135e9962d40fc5f13c676ab557510fe5d1e5071bae9d133dfa5c5
  Stored in directory: /tmp/pip-ephem-wheel-cache-e3ku3dn4/wheels/7b/35/ce/029cd34d6911aad85ccb0c8eb48fe8d426f978c0f73d3944a2
Successfully built repe


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline,BitsAndBytesConfig
from repe import repe_pipeline_registry, WrappedReadingVecModel
import numpy as np
from huggingface_hub import login
login(token="Your Token")

# Register the RepE pipeline
repe_pipeline_registry()

model_name_or_path = "Qwen/Qwen2.5-7B-Instruct"

# 4-bit config to prevent OOM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name_or_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    attn_implementation="eager"
).eval()

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
tokenizer.padding_side = 'left'
tokenizer.pad_token = tokenizer.unk_token if tokenizer.pad_token is None else tokenizer.pad_token

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
dog_train_data = [
    # Original 4 pairs
    "This is a photo of a dog.", "This is a photo of a cat.",
    "The dog sat on the rug.", "The cat sat on the rug.",
    "I own a pet dog.", "I own a pet bird.",
    "The animal is a dog.", "The animal is a hamster.",

    "The dog fetched the ball from the pond.", "The child fetched the ball from the pond.",
    "A small dog shivered under the porch.", "A small rabbit shivered under the porch.",
    "The dog herded the animals across the field.", "The shepherd herded the animals across the field.",
    "Dogs splashed playfully in the shallow water.", "Children splashed playfully in the shallow water.",
    "The dog trotted faithfully beside its owner.", "The pony trotted faithfully beside its owner.",
    "A dog burrowed into the pile of warm laundry.", "A kitten burrowed into the pile of warm laundry.",
    "The dog guarded the entrance to the property.", "The sentry guarded the entrance to the property.",
    "Dogs howled at the full moon above the hills.", "Wolves howled at the full moon above the hills.",
    "The dog tracked the scent through the dense forest.", "The detective tracked the scent through the dense forest.",
    "A wet dog shook water all over the living room.", "A wet toddler shook water all over the living room.",

    "A golden retriever playing in the park", "A large cat lounging in the garden",
    "The puppy barked loudly at the mailman", "The bird chirped loudly on the branch",
    "Dogs are known as man's best friend", "Horses are known as noble companions",
    "A German Shepherd guarding the house", "A security camera guarding the house",

    # DeepSeek set 1 (25 pairs)
    "A golden retriever fetching a stick at the beach", "A tabby cat stalking a bird by the fence",
    "The husky pulled the sled through the snow", "The reindeer pulled the sleigh through the snow",
    "My beagle sniffed every corner of the garden", "My rabbit nibbled every plant in the garden",
    "The border collie herded the sheep expertly", "The sheepdog guided the flock across the field",
    "Puppies tumbled over each other in the basket", "Kittens wrestled each other in the cardboard box",
    "The dalmatian trotted alongside the fire truck", "The horse trotted alongside the parade float",
    "A poodle pranced around the dog show ring", "A peacock fanned its feathers in the courtyard",
    "The bulldog snored loudly on the couch", "The cat snored softly on the windowsill",
    "A labrador splashed happily in the lake", "A duck splashed contentedly in the pond",
    "The chihuahua shivered under the blanket", "The hamster burrowed under the wood shavings",
    "Stray dogs roamed the quiet streets at night", "Stray cats wandered the quiet alleys at night",
    "The corgi waddled across the living room floor", "The duck waddled across the frozen pond",
    "A spaniel pointed towards the hidden pheasant", "A hawk circled above the open meadow",
    "The rottweiler stood guard at the gate", "The security guard stood watch at the entrance",
    "Dog walkers gathered at the park every morning", "Bird watchers gathered at the nature reserve",
    "The dachshund burrowed under the garden fence", "The groundhog burrowed under the vegetable patch",
    "A great dane towered over the coffee table", "A pony stood tall beside the fence post",
    "The terrier chased the squirrel up the oak tree", "The cat chased the mouse through the kitchen",
    "Service dogs assisted their owners through the crowd", "Nursing aides assisted the elderly through the hall",
    "The mutt wagged its tail enthusiastically", "The toddler waved their hands enthusiastically",
    "A bloodhound tracked the scent through the woods", "A detective tracked the suspect through the city",
    "The boxer bounced excitedly at the doorbell", "The child bounced excitedly at the doorbell",
    "Puppy training classes filled up quickly this spring", "Swimming lessons filled up quickly this summer",
    "The shih tzu got a fresh grooming at the salon", "The toddler got a fresh haircut at the barber",
    "A pack of sled dogs rested after the long race", "A fleet of trucks rested after the long haul",

    # Your Newest 20 pairs
    "The greyhound sprinted across the racetrack", "The cheetah sprinted across the savannah",
    "A pit bull cuddled with the children on the sofa", "A house cat cuddled with the children on the sofa",
    "The samoyed's white fur blended with the snow", "The arctic fox's white fur blended with the snow",
    "Barking echoed through the suburban neighborhood", "Car horns echoed through the downtown streets",
    "The newfoundlander dove into the freezing water", "The otter dove into the freezing water",
    "A jack russell dug frantically in the flowerbed", "A squirrel dug frantically in the flowerbed",
    "The vizsla pointed at the flock of geese", "The golfer pointed at the distant flag",
    "Chew toys scattered across the living room carpet", "Building blocks scattered across the playroom carpet",
    "The basenji yodeled instead of barking", "The opera singer yodeled flawlessly on stage",
    "A mastiff drooled onto the hardwood floor", "A toddler drooled onto the hardwood floor",
]

# Update label count to 49
dog_labels = [[True, False]] * 53

# 3. Setup RepE Reading Pipeline
rep_token = -1 # Look at the last token's hidden state
hidden_layers = list(range(-1, -model.config.num_hidden_layers, -1))
rep_reading_pipeline = pipeline("rep-reading", model=model, tokenizer=tokenizer)
rep_reading_pipeline.framework = 'pt'

# Extract the "Dog Direction"
rep_reader = rep_reading_pipeline.get_directions(
    dog_train_data,
    rep_token=rep_token,
    hidden_layers=hidden_layers,
    n_difference=1,
    train_labels=dog_labels,
    direction_method='pca',
    direction_finder_kwargs={"n_components": 1},
    batch_size =8
)

In [5]:
from repe.rep_control_reading_vec import WrappedBlock

# 1. Define a patch for the WrappedBlock initialization
original_init = WrappedBlock.__init__

def patched_init(self, block):
    original_init(self, block)
    # Forward the attention_type attribute required by Qwen2
    if hasattr(block, 'attention_type'):
        self.attention_type = block.attention_type
    else:
        # Default for Qwen2 if the attribute is missing
        self.attention_type = "full_attention"

# 2. Apply the patch to the library's class
WrappedBlock.__init__ = patched_init

In [6]:
def format_qwen_prompt(prompt):
    """
    Formats the prompt using Qwen's ChatML template.
    """
    return f"<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"

In [17]:
layer_id = list(range(-12, -24, -1)) # Controlling middle-to-late layers often works best
block_name = "decoder_block"
component_index = 0

wrapped_model = WrappedReadingVecModel(model, tokenizer)
wrapped_model.unwrap()

wrapped_model.wrap_block(layer_id, block_name=block_name)

# Negative coefficient suppresses the concept
coeff = -10
activations = {}
for layer in layer_id:
    # Normalize the direction to prevent model collapse (garbage output)
    direction = rep_reader.directions[layer][component_index]
    direction = direction / np.linalg.norm(direction)
    sign = rep_reader.direction_signs[layer][component_index]

    activations[layer] = torch.tensor(coeff * direction * sign).to(model.device).to(model.dtype)

# 5. Generate with Suppression
prompt = "Tell me about a popular household pet that barks."
prompt = format_qwen_prompt(prompt)
encoded_inputs = tokenizer(prompt, return_tensors='pt').to(model.device)


wrapped_model.set_controller(layer_id, activations, masks=1)


with torch.no_grad():
    outputs = model.generate(
        **encoded_inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )


generation = tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant\n")[-1]

print("Steered Output:", generation)

# Cleanup
wrapped_model.reset()
wrapped_model.unwrap()

Steered Output: One popular household pet that barks is the domestic duck. While often not a traditional "pet” kept in a home due to their larger size, ducks are known for their distinctive and loud quacking call which can be heard far and wide. 

Ducks are known for their beautiful plumage and playful behavior, as well as their ability to produce large amounts of natural fertilizer from their droppings. They are also a popular food source and are hunted by many cultures around the world. However, they can also be problematic when they begin to overpopulate an area or their noise becomes too much for nearby residents.

The quack of a duck is a complex sound produced by the rapid inhalation and exhalation of air through the bird’s lungs. The sound is produced primarily by the male, who uses it to attract mates and defend territory. However, female ducks also quack but their calls are generally less powerful than those of the males.

While ducks may be popular among wildlife enthusiasts 